# Vision GRPO · LaTeX OCR

Train `mlx-community/Qwen3.5-2B-4bit` with image-conditioned GRPO on
[LaTeX OCR](https://huggingface.co/datasets/mlx-community/LaTeX_OCR).
Each image prompt produces a group of sampled responses. Rewards compare them
with the held-out LaTeX target; group-relative advantages train the language adapters.
The target is never included in the model prompt.

Use an Apple Silicon Mac. Install in your notebook kernel environment, then restart:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel pandas
```

Run cells in order. The first load downloads the model and dataset.


In [ ]:
from pathlib import Path
from difflib import SequenceMatcher
import re

import mlx.core as mx
import mlx.optimizers as optim
import pandas as pd
from datasets import load_dataset

from mlx_vlm import load
from mlx_vlm.trainer import Trainer, GRPOTrainingArgs, prepare_model_for_training
from mlx_vlm.trainer.vlm.grpo.rewards import accuracy_reward
from mlx_vlm.trainer.vlm.grpo.trainer import generate_grpo


## 1. Settings

`batch_size` counts prompts and `group_size` counts completions per prompt.
`iters` counts groups of prompt batches; four accumulation steps make an optimizer update.
Rollouts and gradients process one completion at a time to bound memory.
`max_seq_length` includes image tokens, prompt, and completion. Oversized prompts
raise an error instead of truncating image slots. Use smaller images if necessary.

`beta=0` skips reference-model allocation and scoring. For a KL penalty, set
`beta=0.01`; the model cell then loads a separate frozen base.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
DATASET_ID = "mlx-community/LaTeX_OCR"
OUTPUT_DIR = Path("adapters/vlm/grpo_latex_ocr")
VALIDATION_FRACTION = 0.02
MAX_TRAIN_SAMPLES = 256
MAX_EVAL_SAMPLES = 16
SEED = 42
TRAIN_TYPE = "lora"
CHECKPOINT_PATH = None
OCR_PROMPT = (
    "Transcribe the mathematical expression in this image into LaTeX. "
    "Return only the LaTeX expression, without explanations or Markdown fences."
)
# GRPO variants share this trainer; choose their loss normalization below:
#   "grpo": mean over tokens per completion, then mean over completions.
#   "bnpo": divide the summed loss by total completion tokens.
#   "dr_grpo": divide by completion count * max_completion_length.
# Rewards always use group standard-deviation scaling, including "dr_grpo";
# this option selects the denominator, not every change in the Dr. GRPO method.
# "sequence" importance sampling gives GSPO-style ratios; epsilon_high
# sets a DAPO-style upper clipping width (defaults to epsilon when None).
# Each rollout is used once: ratios start at 1 and clipping normally stays 0.
# No multi-epoch PPO or DAPO dynamic resampling is implemented.
# beta controls reference-model KL; beta=0 skips reference scoring.
# advantage_epsilon stabilizes reward scaling; min_p also controls sampling.
args = GRPOTrainingArgs(
    iters=100,
    batch_size=1,
    group_size=4,
    max_completion_length=128,
    max_seq_length=4096,
    temperature=0.8,
    top_p=0.95,
    top_k=20,
    beta=0.0,
    grpo_loss_type="grpo",  # also "bnpo" or "dr_grpo"
    importance_sampling_level="token",  # "sequence" for GSPO-style ratios
    gradient_accumulation_steps=4,
    grad_checkpoint=True,
    logit_chunk_size=32,
    cache_size=8,
    compile=False,
    val_batches=4,
    steps_per_report=10,
    steps_per_eval=50,
    steps_per_save=100,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()


## 2. Prompt-only image rows

Hold out 2% of training data for validation, preserving the original test partition
for final quality assessment. Each row contains `prompt`, `answer`, and optional
`image` or `images`. Text-only GRPO uses the same API: omit the image column and
supply a text question. `messages` can also hold a conversation ending with a user prompt.


In [ ]:
raw_dataset = load_dataset(DATASET_ID)
partitions = raw_dataset["train"].train_test_split(
    test_size=VALIDATION_FRACTION, seed=SEED,
)


def limit_rows(rows, maximum):
    return rows if maximum is None else rows.select(range(min(maximum, len(rows))))


def latex_to_grpo(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Each OCR row needs a non-empty LaTeX target.")
    return {"prompt": OCR_PROMPT, "answer": text}


train_rows = limit_rows(partitions["train"], MAX_TRAIN_SAMPLES).map(
    latex_to_grpo, input_columns=["text"], remove_columns=["text"],
)
eval_rows = limit_rows(partitions["test"], MAX_EVAL_SAMPLES).map(
    latex_to_grpo, input_columns=["text"], remove_columns=["text"],
)
print({"train_rows": len(train_rows), "validation_rows": len(eval_rows)})
print(train_rows.features)


## 3. Rewards

Exact match is sparse. A character-similarity reward provides a denser signal while
the exact-match reward encourages correct final output. These are example heuristics,
not mathematical equivalence checks: validate generated expressions yourself.
Callbacks receive a batch of decoded completions and return one score per response.
`None` means the callback does not apply; at least one callback must score every response.
Identical rewards within a group yield zero policy advantage, so monitor reward spread.


In [ ]:
def normalize_latex(text):
    return re.sub(r"\s+", "", text.strip().removeprefix("```latex").removesuffix("```"))


def latex_similarity_reward(*, completions, answer, **kwargs):
    return [
        SequenceMatcher(None, normalize_latex(target), normalize_latex(completion)).ratio()
        for completion, target in zip(completions, answer)
    ]


reward_funcs = [latex_similarity_reward, accuracy_reward]
reward_weights = [0.8, 0.2]


## 4. Load and prepare language adapters

LoRA freezes the vision tower and quantized base. Processor image features remain
in the rollout and policy inputs. Keep dropout at zero so generation and scoring
use the same policy. To train vision parameters too, explicitly target their modules
using `prepare_model_for_training` or use the CLI's `--train-vision` option.


In [ ]:
model, processor = load(MODEL_ID)
reference_model = load(MODEL_ID)[0] if args.beta > 0 else None
model = prepare_model_for_training(
    model, train_type=TRAIN_TYPE, lora_rank=8, lora_alpha=16,
    lora_dropout=0.0, checkpoint_path=CHECKPOINT_PATH,
)
optimizer = optim.AdamW(
    learning_rate=1e-5,
    weight_decay=0.0,
)

trainer = Trainer(
    model, args, algorithm="grpo", processing_class=processor,
    optimizer=optimizer,
    train_dataset=train_rows, eval_dataset=eval_rows,
    reward_funcs=reward_funcs, reward_weights=reward_weights,
    reference_model=reference_model, seed=SEED,
)
print({"prompt_tokens": trainer.train_dataset[0]["inputs"]["input_ids"].size})


## 5. Train and inspect reports

Rewards run once over complete groups on the CPU. Qwen3.5 projects only completion
hidden states through the vocabulary head, in bounded chunks; gradients synchronize
between completions. Gradient checkpointing reduces transformer activation memory.
The clipped objective consumes each rollout once, before its generating policy updates.
Consequently importance ratios start at one and clipping metrics normally stay zero.


In [ ]:
metrics = trainer.train()
assert mx.isfinite(mx.array(metrics["train_loss"])).item()
reports = pd.DataFrame(trainer.log_history)
reports.tail(5).reindex(columns=[
    "iteration", "train_loss", "train_reward_mean", "train_grouped_reward_std",
    "train_kl", "trained_tokens", "total_vision_tokens", "peak_memory",
])


## 6. Evaluate sampled responses

Evaluation samples fresh response groups and reports reward and KL diagnostics.
GRPO loss alone does not measure accuracy: compare reward means, reward spread,
and actual generations. Sampling introduces variation between evaluations.


In [ ]:
validation = trainer.evaluate()
pd.DataFrame([validation]).reindex(columns=[
    "val_loss", "val_reward_mean", "val_grouped_reward_std", "val_kl",
])


In [ ]:
sample = trainer.eval_dataset[0]
completion_ids, completion_texts, _ = generate_grpo(
    model, getattr(processor, "tokenizer", processor), [sample], args,
)
print("Target:", sample["answer"])
for index, text in enumerate(completion_texts):
    print(f"Sample {index + 1}:", text)


## 7. Save and reload

Keep `weights.safetensors` and `adapter_config.json` together. Training also saves
final weights automatically. Resuming adapters starts a fresh optimizer.


In [ ]:
trainer.save_adapter()
print(Path(args.adapter_file).resolve())


Reload later:

```python
model, processor = load(MODEL_ID)
model = prepare_model_for_training(
    model, train_type=TRAIN_TYPE, checkpoint_path=args.adapter_file,
)
model.eval()
```

See the [training guide](../../docs/training.md) and the
[MLX-LM-LoRA GRPO implementation](https://github.com/Goekdeniz-Guelmez/mlx-lm-lora/blob/main/mlx_lm_lora/trainer/grpo_trainer.py)
that informed the loss and memory optimizations.
